# QCi Dirac-3 Phase 3 - guarded qBraid reproduction
Use **Run -> Run All Cells** (or **Kernel -> Restart Kernel and Run All Cells**). Do not start with an individual middle cell. The scientific reproduction is credential-free, makes no network or hardware call after dependency installation, and never changes the submitted raw QCi evidence. Optional hardware instructions are separated at the end. Dirac samples are not gate-QPU shots.

Verify the release manifest only on a fresh extraction or clean clone. It intentionally fails after reproduction regenerates figures and timing metadata; extract the ZIP again into a new empty directory or use a clean clone before checking release integrity again.

In [ ]:
from pathlib import Path
import os, subprocess, sys

# Reset in-memory gates before any operation that can fail in a reused kernel.
MANIFEST_VERIFIED = False
SETUP_COMPLETED = False

# qBraid may open a nested notebook with either the repository root or the
# notebook directory as the current working directory. Resolve both safely.
required = ('requirements.txt', 'run_all_local.py')
candidates = (Path.cwd().resolve(), Path.cwd().resolve() / 'Source_Code')
ROOT = next((p for p in candidates if all((p / name).is_file() for name in required)), None)
if ROOT is None:
    raise RuntimeError('Cannot locate Source_Code; open this notebook from the cloned repository.')
subprocess.run([sys.executable, str(ROOT / 'scripts/verify_release_manifest.py'),
                '--root', str(ROOT.parent)], check=True)
MANIFEST_VERIFIED = True
os.chdir(ROOT)

# Keep the pinned scientific/figure stack away from qBraid's preinstalled Qiskit/Braket stack.
VENV = Path.home() / 'qci-phase3-judge-venv'
VENV_PYTHON = VENV / 'bin' / 'python'
if not VENV_PYTHON.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(VENV)], check=True)
subprocess.run([str(VENV_PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check',
                '-r', str(ROOT / 'requirements-docs.txt')], check=True)
SETUP_COMPLETED = True
print(f'Source code root: {ROOT}')
print(f'Isolated Python: {VENV_PYTHON}')

In [ ]:
# One submitted Python entry point performs the complete judge reproduction.
if globals().get('SETUP_COMPLETED') is not True:
    raise RuntimeError('Setup did not complete. Restart the kernel and use Run -> Run All Cells; do not run this cell alone.')
REPRODUCTION_COMPLETED = False
# Use a non-interactive plotting backend inside the isolated environment.
judge_env = os.environ.copy()
judge_env['MPLBACKEND'] = 'Agg'
subprocess.run([str(VENV_PYTHON), str(ROOT / 'run_judge_acceptance.py')],
               cwd=ROOT, env=judge_env, check=True)
REPRODUCTION_COMPLETED = True

In [ ]:
if 'ROOT' not in globals():
    raise RuntimeError('Setup did not run. Restart the kernel and use Run -> Run All Cells; do not run this cell alone.')
assert globals().get('MANIFEST_VERIFIED') is True, 'The initial release manifest did not pass.'
assert globals().get('SETUP_COMPLETED') is True, 'Dependency setup did not complete.'
assert globals().get('REPRODUCTION_COMPLETED') is True, 'Reproduction commands did not complete; no acceptance certificate will be written.'
import json
acceptance_path = ROOT / 'results/reproduction_acceptance.json'
acceptance = json.loads(acceptance_path.read_text())
assert acceptance['status'] == 'PASS'
print(f"Verified acceptance certificate: {acceptance_path.relative_to(ROOT)}")

## Optional independent QCi rerun
A live rerun is stochastic and requires the reviewer's own QCi allocation and **QCi API token** from the QCi portal. Enter the portal-issued API token itself—not a QCi account password, qBraid password, qBraid token, or separate short-lived refreshed/access token. Do not put the token in this notebook. Open a qBraid terminal and follow the root README's optional-hardware section, using the hidden `read -rsp` command. Run the three-sample smoke first and inspect allocation before any nine-job evidence rerun. Do not manually use `run_live_dirac3.py --submit`, `--collect`, or `--unlock`; new attempts belong in the isolated judge namespace.